In [1]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [9]:
import os
import sqlite3
import pandas as pd

# 1. Dynamically scan the directory to find the real database file
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    # Skip virtual environments to optimize the scan speed
    if '.venv' in root or 'node_modules' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            full_path = os.path.join(root, file)
            # Ensure the file isn't a 0-byte dummy file created by previous failures
            if os.path.getsize(full_path) > 0:
                real_db_path = full_path
                break

if not real_db_path:
    print("❌ Critical Error: Could not find a valid restaurant.db with data.")
    print("Please double check if your template database file is located inside this folder hierarchy.")
else:
    print(f"✨ Successfully found the active database file at:\n👉 {real_db_path}\n")

    # 2. Connect to the confirmed database
    conn = sqlite3.connect(real_db_path)

    # 3. Define the Task 1 query
    query = """
    SELECT
        p.name AS product_name,
        SUM(oi.quantity) AS total_quantity
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_id, p.name
    ORDER BY total_quantity DESC
    LIMIT 10;
    """

    try:
        df_top10 = pd.read_sql_query(query, conn)
        print("✅ Task 1 Data Frame Extracted:")
        print(df_top10)

        # Save a snippet locally for analysis
        df_top10.to_csv(os.path.join(BASE_DIR, 'top_10_products.csv'), index=False)

    except Exception as e:
        print(f"An unexpected query execution error occurred: {e}")
    finally:
        conn.close()


❌ Critical Error: Could not find a valid restaurant.db with data.
Please double check if your template database file is located inside this folder hierarchy.


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [3]:
import os
import sqlite3
import pandas as pd

# 1. Dynamically locate the correct absolute path
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
DB_PATH = os.path.join(BASE_DIR, "restaurant.db")

print(f"Targeting Database File at: {DB_PATH}")

if not os.path.exists(DB_PATH):
    # If it's stored inside a subfolder like 'db' or 'data', find it
    print("❌ Warning: 'restaurant.db' was not found in the root directory.")
    print("Searching subdirectories for the database file...")
    for root, dirs, files in os.walk(BASE_DIR):
        if "restaurant.db" in files:
            DB_PATH = os.path.join(root, "restaurant.db")
            print(f"👉 Found real file here: {DB_PATH}")
            break

# 2. Connect directly to the confirmed path
conn = sqlite3.connect(DB_PATH)

# Let's run a safety check to see if tables actually exist here
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = [t[0] for t in cursor.fetchall()]
print(f"Actual tables found in this file: {tables}")

if "order_items" in tables:
    # 3. If valid, proceed with Task 1 Data Aggregation
    query = """
    SELECT
        p.name AS product_name,
        SUM(oi.quantity) AS total_quantity
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_id, p.name
    ORDER BY total_quantity DESC
    LIMIT 10;
    """
    df_top10 = pd.read_sql_query(query, conn)
    print("\n✅ Task 1 Data Extracted Successfully:")
    print(df_top10)

    # Save a CSV snippet for your records
    df_top10.to_csv(os.path.join(BASE_DIR, 'top_10_products.csv'), index=False)
else:
    print("❌ The table 'order_items' is still missing. Verify if your project initialization scripts or seeds were executed!")

conn.close()


Targeting Database File at: C:\Mate academy\git-1\py-restaurant-data-analysis\restaurant.db
❌ Warning: 'restaurant.db' was not found in the root directory.
Searching subdirectories for the database file...
👉 Found real file here: C:\Mate academy\git-1\py-restaurant-data-analysis\app\restaurant.db
Actual tables found in this file: []
❌ The table 'order_items' is still missing. Verify if your project initialization scripts or seeds were executed!


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [12]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Dynamically scan the repository to find the actual populated database file
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    if '.venv' in root or '.git' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            test_path = os.path.join(root, file)
            # Ensure the database contains data (is greater than 0 bytes)
            if os.path.getsize(test_path) > 0:
                real_db_path = test_path
                break

if not real_db_path:
    print("❌ Error: Could not locate a valid 'restaurant.db' containing data.")
    print("👉 Check your project directory. Is there a setup/init script you need to execute first?")
else:
    print(f"✨ Found active database template at:\n🔗 {real_db_path}\n")

    # 2. Establish SQLite connection
    conn = sqlite3.connect(real_db_path)

    # 3. TASK 0: Clean Join & Extract
    query_all = """
    SELECT
        oi.order_id,
        oi.product_id,
        oi.quantity,
        o.date AS order_date,
        p.name AS product_name,
        p.price AS product_price
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    """

    try:
        # Load unified dataset
        df = pd.read_sql_query(query_all, conn)
        print(f"✅ TASK 0 Complete: Combined dataset contains {len(df)} rows.")

        # Save to local CSV file as requested by Task 0 instructions
        csv_out_path = os.path.join(BASE_DIR, 'combined_restaurant_data.csv')
        df.to_csv(csv_out_path, index=False)
        print(f"📁 Extracted CSV exported to: {csv_out_path}")

        # ==========================================
        # TASK 1: Top 10 Most Popular By Quantity
        # ==========================================
        top10_qty = (
            df.groupby('product_name')['quantity']
            .sum()
            .reset_index()
            .sort_values(by='quantity', ascending=False)
            .head(10)
        )

        # Plot Task 1
        plt.figure(figsize=(8, 6))
        plt.pie(top10_qty['quantity'], labels=top10_qty['product_name'], autopct='%1.1f%%', startangle=140)
        plt.title('Top 10 Most Popular Products (By Quantity Sold)', fontsize=12, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        plt.show()

        # ==========================================
        # TASK 2: Top 10 High Earners By Revenue
        # ==========================================
        # Calculate 'Item Price' (Product Price * Quantity)
        df['Item Price'] = df['product_price'] * df['quantity']

        top10_revenue = (
            df.groupby('product_name')['Item Price']
            .sum()
            .reset_index()
            .sort_values(by='Item Price', ascending=False)
            .head(10)
        )

        # Plot Task 2
        plt.figure(figsize=(8, 6))
        plt.pie(top10_revenue['Item Price'], labels=top10_revenue['product_name'], autopct='%1.1f%%', startangle=140)
        plt.title('Top 10 Revenue Generating Products (By Revenue Share)', fontsize=12, fontweight='bold')
        plt.axis('equal')
        plt.tight_layout()
        plt.show()

    except Exception as error:
        print(f"❌ Query compilation failed: {error}")
    finally:
        conn.close()


❌ Error: Could not locate a valid 'restaurant.db' containing data.
👉 Check your project directory. Is there a setup/init script you need to execute first?


# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [14]:
import os

base_dir = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
print("🔍 Searching your project folder for 'restaurant.db'...")

found = False
for root, dirs, files in os.walk(base_dir):
    # Ignore virtual environment folders to make it fast
    if '.venv' in root or '.git' in root:
        continue
    for file in files:
        if 'restaurant' in file.lower() or file.endswith('.db'):
            full_path = os.path.join(root, file)
            size = os.path.getsize(full_path)
            print(f"👉 Found file: {full_path} ({size} bytes)")
            found = True

if not found:
    print("❌ No database file found anywhere in the directory tree!")
    print("Check if you have files like 'seed.sql', 'db.sqlite', or a script to run first.")


🔍 Searching your project folder for 'restaurant.db'...
👉 Found file: C:\Mate academy\git-1\py-restaurant-data-analysis\restaurant.db (0 bytes)
👉 Found file: C:\Mate academy\git-1\py-restaurant-data-analysis\.idea\py-restaurant-data-analysis.iml (426 bytes)
👉 Found file: C:\Mate academy\git-1\py-restaurant-data-analysis\app\restaurant.db (0 bytes)


import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Dynamically locate the active database file with data (>0 bytes)
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    if '.venv' in root or '.git' in root or 'node_modules' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            test_path = os.path.join(root, file)
            if os.path.getsize(test_path) > 0:
                real_db_path = test_path
                break

if not real_db_path:
    print("❌ Error: Could not locate a valid 'restaurant.db' containing data.")
    print("Please double check if your template database file was downloaded or seeded correctly.")
else:
    print(f"✨ Connected to valid database at: {real_db_path}")

    # 2. Extract and join raw data (Task 0 setup)
    conn = sqlite3.connect(real_db_path)
    query_all = """
    SELECT
        oi.quantity,
        o.date AS order_date,
        p.price AS product_price
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    """

    try:
        df = pd.read_sql_query(query_all, conn)
        conn.close()

        # 3. Calculate Item Price and Datetime parameters
        df['Item Price'] = df['product_price'] * df['quantity']
        df['order_date'] = pd.to_datetime(df['order_date'])

        # =======================================================
        # TASK 3 RECAP: Hourly Breakdown
        # =======================================================
        df['Order Hour'] = df['order_date'].dt.hour
        hourly_income = df.groupby('Order Hour')['Item Price'].sum().reindex(range(24), fill_value=0).reset_index()

        plt.figure(figsize=(10, 4))
        plt.bar(hourly_income['Order Hour'], hourly_income['Item Price'], color='royalblue', edgecolor='black')
        plt.title('Total Income by Order Hour (Task 3)', fontsize=12, fontweight='bold')
        plt.xlabel('Hour of Day (0-23)')
        plt.ylabel('Total Income ($)')
        plt.xticks(range(24))
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # =======================================================
        # TASK 4: Day of the Week Breakdown
        # =======================================================
        # dt.day_name() returns 'Monday', 'Tuesday', etc.
        df['Day of Week'] = df['order_date'].dt.day_name()

        # Group and aggregate income by day
        weekly_income = df.groupby('Day of Week')['Item Price'].sum().reset_index()

        # Explicit order mapping from Monday to Sunday
        days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        weekly_income['Day of Week'] = pd.Categorical(weekly_income['Day of Week'], categories=days_order, ordered=True)
        weekly_income = weekly_income.sort_values('Day of Week')

        print("\n--- Total Restaurant Income by Day of Week ---")
        print(weekly_income.to_string(index=False))

        # Plot Task 4 Bar Chart
        plt.figure(figsize=(10, 4))
        plt.bar(
            weekly_income['Day of Week'],
            weekly_income['Item Price'],
            color='emerald' if 'emerald' in plt.cm.datad else 'mediumseagreen',
            edgecolor='black',
            alpha=0.85
        )
        plt.title('Total Restaurant Income by Day of the Week (Task 4)', fontsize=12, fontweight='bold')
        plt.xlabel('Day of the Week', fontsize=10)
        plt.ylabel('Total Income ($)', fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error during query processing: {e}")
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Dynamically locate the active database file with data (>0 bytes)
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    if '.venv' in root or '.git' in root or 'node_modules' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            test_path = os.path.join(root, file)
            if os.path.getsize(test_path) > 0:
                real_db_path = test_path
                break

if not real_db_path:
    print("❌ Error: Could not locate a valid 'restaurant.db' containing data.")
    print("Please double check if your template database file was downloaded or seeded correctly.")
else:
    print(f"✨ Connected to valid database at: {real_db_path}")

    # 2. Extract and join raw data (Task 0 setup)
    conn = sqlite3.connect(real_db_path)
    query_all = """
    SELECT
        oi.quantity,
        o.date AS order_date,
        p.price AS product_price
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    """

    try:
        df = pd.read_sql_query(query_all, conn)
        conn.close()

        # 3. Calculate Item Price and Datetime parameters
        df['Item Price'] = df['product_price'] * df['quantity']
        df['order_date'] = pd.to_datetime(df['order_date'])

        # =======================================================
        # TASK 3 RECAP: Hourly Breakdown
        # =======================================================
        df['Order Hour'] = df['order_date'].dt.hour
        hourly_income = df.groupby('Order Hour')['Item Price'].sum().reindex(range(24), fill_value=0).reset_index()

        plt.figure(figsize=(10, 4))
        plt.bar(hourly_income['Order Hour'], hourly_income['Item Price'], color='royalblue', edgecolor='black')
        plt.title('Total Income by Order Hour (Task 3)', fontsize=12, fontweight='bold')
        plt.xlabel('Hour of Day (0-23)')
        plt.ylabel('Total Income ($)')
        plt.xticks(range(24))
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # =======================================================
        # TASK 4: Day of the Week Breakdown
        # =======================================================
        # dt.day_name() returns 'Monday', 'Tuesday', etc.
        df['Day of Week'] = df['order_date'].dt.day_name()

        # Group and aggregate income by day
        weekly_income = df.groupby('Day of Week')['Item Price'].sum().reset_index()

        # Explicit order mapping from Monday to Sunday
        days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        weekly_income['Day of Week'] = pd.Categorical(weekly_income['Day of Week'], categories=days_order, ordered=True)
        weekly_income = weekly_income.sort_values('Day of Week')

        print("\n--- Total Restaurant Income by Day of Week ---")
        print(weekly_income.to_string(index=False))

        # Plot Task 4 Bar Chart
        plt.figure(figsize=(10, 4))
        plt.bar(
            weekly_income['Day of Week'],
            weekly_income['Item Price'],
            color='emerald' if 'emerald' in plt.cm.datad else 'mediumseagreen',
            edgecolor='black',
            alpha=0.85
        )
        plt.title('Total Restaurant Income by Day of the Week (Task 4)', fontsize=12, fontweight='bold')
        plt.xlabel('Day of the Week', fontsize=10)
        plt.ylabel('Total Income ($)', fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error during query processing: {e}")
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Dynamically locate the active database file with data (>0 bytes)
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    if '.venv' in root or '.git' in root or 'node_modules' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            test_path = os.path.join(root, file)
            if os.path.getsize(test_path) > 0:
                real_db_path = test_path
                break

if not real_db_path:
    print("❌ Error: Could not locate a valid 'restaurant.db' containing data.")
    print("Please double check if your template database file was downloaded or seeded correctly.")
else:
    print(f"✨ Connected to valid database at: {real_db_path}")

    # 2. Extract and join raw data (Task 0 setup)
    conn = sqlite3.connect(real_db_path)
    query_all = """
    SELECT
        oi.quantity,
        o.date AS order_date,
        p.price AS product_price
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    """

    try:
        df = pd.read_sql_query(query_all, conn)
        conn.close()

        # 3. Calculate Item Price and Datetime parameters
        df['Item Price'] = df['product_price'] * df['quantity']
        df['order_date'] = pd.to_datetime(df['order_date'])

        # =======================================================
        # TASK 3 RECAP: Hourly Breakdown
        # =======================================================
        df['Order Hour'] = df['order_date'].dt.hour
        hourly_income = df.groupby('Order Hour')['Item Price'].sum().reindex(range(24), fill_value=0).reset_index()

        plt.figure(figsize=(10, 4))
        plt.bar(hourly_income['Order Hour'], hourly_income['Item Price'], color='royalblue', edgecolor='black')
        plt.title('Total Income by Order Hour (Task 3)', fontsize=12, fontweight='bold')
        plt.xlabel('Hour of Day (0-23)')
        plt.ylabel('Total Income ($)')
        plt.xticks(range(24))
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # =======================================================
        # TASK 4: Day of the Week Breakdown
        # =======================================================
        # dt.day_name() returns 'Monday', 'Tuesday', etc.
        df['Day of Week'] = df['order_date'].dt.day_name()

        # Group and aggregate income by day
        weekly_income = df.groupby('Day of Week')['Item Price'].sum().reset_index()

        # Explicit order mapping from Monday to Sunday
        days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        weekly_income['Day of Week'] = pd.Categorical(weekly_income['Day of Week'], categories=days_order, ordered=True)
        weekly_income = weekly_income.sort_values('Day of Week')

        print("\n--- Total Restaurant Income by Day of Week ---")
        print(weekly_income.to_string(index=False))

        # Plot Task 4 Bar Chart
        plt.figure(figsize=(10, 4))
        plt.bar(
            weekly_income['Day of Week'],
            weekly_income['Item Price'],
            color='emerald' if 'emerald' in plt.cm.datad else 'mediumseagreen',
            edgecolor='black',
            alpha=0.85
        )
        plt.title('Total Restaurant Income by Day of the Week (Task 4)', fontsize=12, fontweight='bold')
        plt.xlabel('Day of the Week', fontsize=10)
        plt.ylabel('Total Income ($)', fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error during query processing: {e}")
# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [1]:
import os
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# 1. Dynamically locate the active database file with data (>0 bytes)
BASE_DIR = r"C:\Mate academy\git-1\py-restaurant-data-analysis"
real_db_path = None

for root, dirs, files in os.walk(BASE_DIR):
    if '.venv' in root or '.git' in root or 'node_modules' in root:
        continue
    for file in files:
        if file == "restaurant.db":
            test_path = os.path.join(root, file)
            if os.path.getsize(test_path) > 0:
                real_db_path = test_path
                break

if not real_db_path:
    print("❌ Error: Could not locate a valid 'restaurant.db' containing data.")
    print("Please double check if your template database file was downloaded or seeded correctly.")
else:
    print(f"✨ Connected to valid database at: {real_db_path}")

    # 2. Extract and join raw data (Task 0 setup)
    conn = sqlite3.connect(real_db_path)
    query_all = """
    SELECT
        oi.quantity,
        o.date AS order_date,
        p.price AS product_price
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    """

    try:
        df = pd.read_sql_query(query_all, conn)
        conn.close()

        # 3. Calculate Item Price and Datetime parameters
        df['Item Price'] = df['product_price'] * df['quantity']
        df['order_date'] = pd.to_datetime(df['order_date'])

        # =======================================================
        # TASK 3 RECAP: Hourly Breakdown
        # =======================================================
        df['Order Hour'] = df['order_date'].dt.hour
        hourly_income = df.groupby('Order Hour')['Item Price'].sum().reindex(range(24), fill_value=0).reset_index()

        plt.figure(figsize=(10, 4))
        plt.bar(hourly_income['Order Hour'], hourly_income['Item Price'], color='royalblue', edgecolor='black')
        plt.title('Total Income by Order Hour (Task 3)', fontsize=12, fontweight='bold')
        plt.xlabel('Hour of Day (0-23)')
        plt.ylabel('Total Income ($)')
        plt.xticks(range(24))
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

        # =======================================================
        # TASK 4: Day of the Week Breakdown
        # =======================================================
        # dt.day_name() returns 'Monday', 'Tuesday', etc.
        df['Day of Week'] = df['order_date'].dt.day_name()

        # Group and aggregate income by day
        weekly_income = df.groupby('Day of Week')['Item Price'].sum().reset_index()

        # Explicit order mapping from Monday to Sunday
        days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        weekly_income['Day of Week'] = pd.Categorical(weekly_income['Day of Week'], categories=days_order, ordered=True)
        weekly_income = weekly_income.sort_values('Day of Week')

        print("\n--- Total Restaurant Income by Day of Week ---")
        print(weekly_income.to_string(index=False))

        # Plot Task 4 Bar Chart
        plt.figure(figsize=(10, 4))
        plt.bar(
            weekly_income['Day of Week'],
            weekly_income['Item Price'],
            color='emerald' if 'emerald' in plt.cm.datad else 'mediumseagreen',
            edgecolor='black',
            alpha=0.85
        )
        plt.title('Total Restaurant Income by Day of the Week (Task 4)', fontsize=12, fontweight='bold')
        plt.xlabel('Day of the Week', fontsize=10)
        plt.ylabel('Total Income ($)', fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ Error during query processing: {e}")


❌ Error: Could not locate a valid 'restaurant.db' containing data.
Please double check if your template database file was downloaded or seeded correctly.
